#EXTRAER CARACTERISTICAS BASE DE FRUTAS (NARANJAS)

-------

Carlos Francisco Montoya Urzua

Lunes, 25 de mayo de 2026

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
### Código para extraer características de múltiples imágenes y guardar en CSV

import cv2
from google.colab.patches import cv2_imshow
import os
import numpy as np
import pandas as pd
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import LabelEncoder

# Función para extraer características
def extraer_caracteristicas(ruta_imagen):
    # Cargar imagen y convertir a HSV y escala de grises
    imagen_bgr = cv2.imread(ruta_imagen)
    cv2_imshow(imagen_bgr)
    imagen_hsv = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2HSV)
    imagen_gray = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2GRAY)

    # Histograma de color (HSV)
    hist_h = cv2.calcHist([imagen_hsv], [0], None, [8], [0, 180]).flatten()
    hist_s = cv2.calcHist([imagen_hsv], [1], None, [8], [0, 256]).flatten()
    hist_v = cv2.calcHist([imagen_hsv], [2], None, [8], [0, 256]).flatten()
    hist_color = np.concatenate((hist_h, hist_s, hist_v))
    hist_color = hist_color / np.sum(hist_color)  # normalización

    # Características de forma: contornos
    _, binaria = cv2.threshold(imagen_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contornos, _ = cv2.findContours(binaria, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contornos:
        mayor_contorno = max(contornos, key=cv2.contourArea)
        area = cv2.contourArea(mayor_contorno)
        perimetro = cv2.arcLength(mayor_contorno, True)
        momentos = cv2.moments(mayor_contorno)
        if momentos['m00']==0: momentos['m00']=1
        cx = int(momentos['m10']/momentos['m00']) if momentos['m00'] != 0 else 0
        cy = int(momentos['m01']/momentos['m00']) if momentos['m00'] != 0 else 0
    else:
        area = perimetro = cx = cy = 0

    forma = np.array([area, perimetro, cx, cy], dtype=np.float32)

    # Características de textura: LBP
    lbp = local_binary_pattern(imagen_gray, P=8, R=1, method='uniform')
    (hist_lbp, _) = np.histogram(lbp.ravel(), bins=np.arange(0, 10), range=(0, 9))
    hist_lbp = hist_lbp.astype("float")
    hist_lbp /= (hist_lbp.sum() + 1e-7)

    # Características de puntos clave: ORB
    orb = cv2.ORB_create()
    keypoints, descriptores = orb.detectAndCompute(imagen_gray, None)
    num_keypoints = len(keypoints)
    orb_feature = np.array([num_keypoints], dtype=np.float32)

    # Concatenar todas las características
    vector_final = np.concatenate([hist_color, forma, hist_lbp, orb_feature])

    return vector_final

# Función para recorrer el directorio, extraer características y guardarlas en un CSV
def guardar_caracteristicas_en_csv(directorio_imagenes, archivo_salida):
    # Crear lista para almacenar características y etiquetas
    caracteristicas = []
    etiquetas = []

    # Recorrer el directorio
    for imagen_nombre in os.listdir(directorio_imagenes):
        ruta_imagen = os.path.join(directorio_imagenes, imagen_nombre)

        if not os.path.isfile(ruta_imagen):
            continue

        # Extraer las características
        vector_caracteristicas = extraer_caracteristicas(ruta_imagen)

        # Etiqueta: podemos usar el nombre del archivo o un valor que tú definas
        etiqueta = imagen_nombre.split('.')[0]  # Esto asume que la etiqueta es el nombre de archivo sin la extensión

        # Guardar las características y la etiqueta
        caracteristicas.append(vector_caracteristicas)
        etiquetas.append(etiqueta)

    # Convertir la lista de características a un arreglo de NumPy
    X = np.array(caracteristicas)

    # Convertir las etiquetas a números usando LabelEncoder (si las etiquetas no son numéricas)
    le = LabelEncoder()
    y = le.fit_transform(etiquetas)

    # Crear un DataFrame de Pandas y guardarlo como archivo CSV
    df = pd.DataFrame(X)
    df['etiqueta'] = y
    df.to_csv(archivo_salida, index=False)
    print(f"Archivo CSV guardado en: {archivo_salida}")

# Ejecutar el proceso para un conjunto de imágenes

directorio_imagenes = '/content/drive/MyDrive/Colab Notebooks/CARACTERISTICAS_FRUTAS/naranjas'  # Cambia esto a la ruta de tus imágenes
archivo_salida = '/content/drive/MyDrive/Colab Notebooks/CARACTERISTICAS_FRUTAS/caracteristicas_imagenes.csv'  # Nombre del archivo CSV de salida

guardar_caracteristicas_en_csv(directorio_imagenes, archivo_salida)

Output hidden; open in https://colab.research.google.com to view.